In [1]:
import numpy as np
import pandas as pd

from linearmodels import PooledOLS          # Pooled model
from linearmodels import RandomEffects      # Random-effect model
from linearmodels import PanelOLS           # Fixed-effect model
from linearmodels import FirstDifferenceOLS # First difference model

from linearmodels.panel import compare      # Compare the results of multiple models
from statsmodels.api import add_constant    # for matrices of regression design

In [2]:
df = pd.read_csv('Gasoline.csv')
df.head()

,country,year,lgaspcar,lincomep,lrpmg,lcarpcap
0,AUSTRIA,1960,4.173244,-6.474277,-0.334548,-9.766840
1,AUSTRIA,1961,4.100989,-6.426006,-0.351328,-9.608622
2,AUSTRIA,1962,4.073177,-6.407308,-0.379518,-9.457257
3,AUSTRIA,1963,4.059509,-6.370679,-0.414251,-9.343155
4,AUSTRIA,1964,4.037689,-6.322247,-0.445335,-9.237739


In [3]:
panel_df = df.set_index(['country', 'year'])
panel_df.head()

lgaspcar  lincomep     lrpmg  lcarpcap
country year                                        
AUSTRIA 1960  4.173244 -6.474277 -0.334548 -9.766840
        1961  4.100989 -6.426006 -0.351328 -9.608622
        1962  4.073177 -6.407308 -0.379518 -9.457257
        1963  4.059509 -6.370679 -0.414251 -9.343155
        1964  4.037689 -6.322247 -0.445335 -9.237739

In [4]:
mod_pl = PooledOLS.from_formula(formula='lgaspcar~1+lincomep+lrpmg+lcarpcap', data=panel_df)
mod_re = RandomEffects.from_formula(formula='lgaspcar~1+lincomep+lrpmg+lcarpcap', data=panel_df)
mod_fe = PanelOLS.from_formula(formula='lgaspcar~1+lincomep+lrpmg+lcarpcap+EntityEffects', data=panel_df)
mod_fd = FirstDifferenceOLS.from_formula(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=panel_df)

res_pl = mod_pl.fit(cov_type='clustered', cluster_entity=True)
res_re = mod_re.fit(cov_type='clustered', cluster_entity=True)
res_fe = mod_fe.fit(cov_type='clustered', cluster_entity=True)
res_fd = mod_fd.fit(cov_type='clustered', cluster_entity=True)

compare({'Pool': res_pl, 'RE': res_re, 'FE': res_fe, 'FD':res_fd}, stars=True)

,Pool,RE,FE,FD
Dep. Variable,lgaspcar,lgaspcar,lgaspcar,lgaspcar
Estimator,PooledOLS,RandomEffects,PanelOLS,FirstDifferenceOLS
No. Observations,342,342,342,324
Cov. Est.,Clustered,Clustered,Clustered,Clustered
R-squared,0.8549,0.8293,0.8396,0.5117
R-Squared (Within),0.7435,0.8363,0.8396,0.8231
R-Squared (Between),0.8771,0.7099,0.5424,0.8814
R-Squared (Overall),0.8549,0.7309,0.5917,0.8813
F-statistic,664.00,547.40,560.09,112.11
P-value (F-stat),0.0000,0.0000,0.0000,0.0000


In [5]:
mod_pl = PooledOLS.from_formula(formula='lgaspcar~1+lincomep+lrpmg+lcarpcap', data=panel_df)
mod_re = RandomEffects.from_formula(formula='lgaspcar~1+lincomep+lrpmg+lcarpcap', data=panel_df)
mod_fe = PanelOLS.from_formula(formula='lgaspcar~1+lincomep+lrpmg+lcarpcap+EntityEffects', data=panel_df)
mod_fd = FirstDifferenceOLS.from_formula(formula='lgaspcar~lincomep+lrpmg+lcarpcap', data=panel_df)

res_pl = mod_pl.fit(cov_type='kernel')
res_re = mod_re.fit(cov_type='kernel')
res_fe = mod_fe.fit(cov_type='kernel')
res_fd = mod_fd.fit(cov_type='kernel')

compare({'Pool': res_pl, 'RE': res_re, 'FE': res_fe, 'FD':res_fd}, stars=True, precision='std_errors')

,Pool,RE,FE,FD
Dep. Variable,lgaspcar,lgaspcar,lgaspcar,lgaspcar
Estimator,PooledOLS,RandomEffects,PanelOLS,FirstDifferenceOLS
No. Observations,342,342,342,324
Cov. Est.,Driscoll-Kraay,Driscoll-Kraay,Driscoll-Kraay,Driscoll-Kraay
R-squared,0.8549,0.8293,0.8396,0.5117
R-Squared (Within),0.7435,0.8363,0.8396,0.8231
R-Squared (Between),0.8771,0.7099,0.5424,0.8814
R-Squared (Overall),0.8549,0.7309,0.5917,0.8813
F-statistic,664.00,547.40,560.09,112.11
P-value (F-stat),0.0000,0.0000,0.0000,0.0000


In [6]:
mod = PanelOLS.from_formula(formula='lgaspcar~1+lincomep+lrpmg+lcarpcap+EntityEffects', data=panel_df)
res_ols = mod.fit()
res_hc = mod.fit(cov_type='robust')
res_ab = mod.fit(cov_type='clustered', cluster_entity=True)
res_dk = mod.fit(cov_type='kernel')

compare({'OLS': res_ols, 'HC': res_hc, 'AB': res_ab, 'DK': res_dk}, stars=True, precision='std_errors')

,OLS,HC,AB,DK
Dep. Variable,lgaspcar,lgaspcar,lgaspcar,lgaspcar
Estimator,PanelOLS,PanelOLS,PanelOLS,PanelOLS
No. Observations,342,342,342,342
Cov. Est.,Unadjusted,Robust,Clustered,Driscoll-Kraay
R-squared,0.8396,0.8396,0.8396,0.8396
R-Squared (Within),0.8396,0.8396,0.8396,0.8396
R-Squared (Between),0.5424,0.5424,0.5424,0.5424
R-Squared (Overall),0.5917,0.5917,0.5917,0.5917
F-statistic,560.09,560.09,560.09,560.09
P-value (F-stat),0.0000,0.0000,0.0000,0.0000


In [7]:
res_dk

Dep. Variable:,lgaspcar,R-squared:,0.8396
Estimator:,PanelOLS,R-squared (Between):,0.5424
No. Observations:,342,R-squared (Within):,0.8396
Date:,"Wed, Mar 18 2026",R-squared (Overall):,0.5917
Time:,20:14:27,Log-likelihood,340.33
Cov. Estimator:,Driscoll-Kraay,,
,,F-statistic:,560.09
Entities:,18,P-value,0.0000
Avg Obs:,19.000,Distribution:,"F(3,321)"
Min Obs:,19.000,,
Max Obs:,19.000,F-statistic (robust):,3175.7


# Листок 2
# 2.1 Модель 1. Результаты оценивания

In [8]:
df = pd.read_csv('Wages.csv')
df.head()

,exp,wks,bluecol,ind,south,smsa,married,sex,union,ed,black,lwage,id,time
0,3,32,no,0,yes,no,yes,male,no,9,no,5.56068,1,1
1,4,43,no,0,yes,no,yes,male,no,9,no,5.72031,1,2
2,5,40,no,0,yes,no,yes,male,no,9,no,5.99645,1,3
3,6,39,no,0,yes,no,yes,male,no,9,no,5.99645,1,4
4,7,42,no,1,yes,no,yes,male,no,9,no,6.06146,1,5


In [9]:
panel_df = df.set_index(['id', 'time'])
panel_df.head()

exp  wks bluecol  ind south smsa married   sex union  ed black  \
id time                                                                   
1  1       3   32      no    0   yes   no     yes  male    no   9    no   
   2       4   43      no    0   yes   no     yes  male    no   9    no   
   3       5   40      no    0   yes   no     yes  male    no   9    no   
   4       6   39      no    0   yes   no     yes  male    no   9    no   
   5       7   42      no    1   yes   no     yes  male    no   9    no   

           lwage  
id time           
1  1     5.56068  
   2     5.72031  
   3     5.99645  
   4     5.99645  
   5     6.06146

In [10]:
mod = PooledOLS.from_formula(formula='lwage~1+ed+exp+I(exp**2)+wks', data=panel_df)
res_ols = mod.fit()
res_ab = mod.fit(cov_type='clustered', cluster_entity=True)
res_dk = mod.fit(cov_type='kernel', kernel = "parzen")

compare({'OLS': res_ols, 'AB': res_ab, 'DK': res_dk}, stars=True, precision='std_errors')

,OLS,AB,DK
Dep. Variable,lwage,lwage,lwage
Estimator,PooledOLS,PooledOLS,PooledOLS
No. Observations,4165,4165,4165
Cov. Est.,Unadjusted,Clustered,Driscoll-Kraay
R-squared,0.2836,0.2836,0.2836
R-Squared (Within),0.1957,0.1957,0.1957
R-Squared (Between),0.3163,0.3163,0.3163
R-Squared (Overall),0.2836,0.2836,0.2836
F-statistic,411.62,411.62,411.62
P-value (F-stat),0.0000,0.0000,0.0000


In [12]:
mod = RandomEffects.from_formula(formula='lwage~1+ed+exp+I(exp**2)+wks', data=panel_df)
res_ols = mod.fit()
res_ab = mod.fit(cov_type='clustered', cluster_entity=True)
res_dk = mod.fit(cov_type='kernel', kernel = "parzen")

compare({'OLS': res_ols, 'AB': res_ab, 'DK': res_dk}, stars=True, precision='std_errors')

,OLS,AB,DK
Dep. Variable,lwage,lwage,lwage
Estimator,RandomEffects,RandomEffects,RandomEffects
No. Observations,4165,4165,4165
Cov. Est.,Unadjusted,Clustered,Driscoll-Kraay
R-squared,0.4200,0.4200,0.4200
R-Squared (Within),0.5487,0.5487,0.5487
R-Squared (Between),-1.1061,-1.1061,-1.1061
R-Squared (Overall),-0.6571,-0.6571,-0.6571
F-statistic,753.01,753.01,753.01
P-value (F-stat),0.0000,0.0000,0.0000
